# JuliaBUGS in Colab

Fitting a graphical model with **JuliaBUGS**, through the `mcmc` command line tool.

Set the model and the sampler below, then run every cell in order. The notebook fits the model, checks that it converged, draws the posterior, and packages the run so it can be opened in the report app.

## Model

In the editor, open the **Run** tab and press **Copy graph**.
Paste it between the quotes below, then run every cell in order.

In [ ]:
GRAPH = raw"""

"""

isempty(strip(GRAPH)) && error("Paste your graph above, then run this cell again.")

write("model.json", GRAPH)

name = match(r"\"name\": \"(.*?)\"", GRAPH)
println("model: ", name === nothing ? "unnamed" : name[1])

## Settings

Change these and run again from here.

In [ ]:
CHAINS = 4
DRAWS = 1000
WARMUP = 1000
SEED = 42

## Install

`mcmc` is a self-contained binary. `mcmc setup` then installs the JuliaBUGS toolchain, which takes a few minutes on a fresh Colab runtime.

In [ ]:
run(pipeline(`curl -fsSL https://mcmcjs.github.io/install.sh`, `sh`))

ENV["PATH"] = joinpath(homedir(), ".local", "bin") * ":" * ENV["PATH"]
run(`mcmc --version`);

In [ ]:
run(`mcmc setup --engine julia`);

## The JuliaBUGS model

What the graph becomes as code, and the spec that runs it.

In [ ]:
run(`mcmc convert model.json`)
print(read("model.jl", String))

## Fit

`mcmc run` samples, checks convergence, and records the run.

In [ ]:
seed = SEED === nothing ? String[] : ["--seed", string(SEED)]

# A run that did not converge exits 2, which the next cells report on rather than stop at.
run(ignorestatus(`mcmc run model.toml --chains $CHAINS --draws $DRAWS --warmup $WARMUP $seed`));

## Did it converge?

R-hat near 1 and a healthy effective sample size for every parameter. `mcmc diagnose` exits non-zero when it did not, so read this before the posterior.

In [ ]:
run(`mcmc summary`)
run(ignorestatus(`mcmc diagnose`));

## Plots

Traces and ranks show the chains mixing, densities and the forest plot show the posterior itself.

In [ ]:
for kind in ["trace","density","forest","rank"]
    run(`mcmc plot --kind $kind --format svg -o $kind.svg`)
    println(kind)
    display("image/svg+xml", read("$kind.svg", String))
end

## Open the run in the report app

A run bundle holds the samples, the spec and the diagnostics in one file. Download it from the Files panel on the left and drop it into [the report app](https://mcmcjs.github.io/report/) to explore every parameter.

In [ ]:
run(`mcmc export bundle -o run.mcmcrun.json`);